# Changelog

## V1

Add flipping on x + y axis for all images.

# Improved De-Poisoning: Backbone-Frozen Unlearning + Gradient Ascent

Improvements over the simple fine-tuning baseline:
1. **Backbone + FPN frozen** — poison lives in the detection head, not ResNet
2. **More iterations (150)** with warmup + cosine LR decay
3. **Gradient ascent phase** — actively pushes model away from poisoned behaviour before the empty-label fine-tune
4. **Weight averaging** — averages GA-phase and fine-tune-phase weights to stabilise predictions

**Workflow**
1. Install Detectron2
2. Load poisoned model
3. Phase A — Gradient Ascent on poisoned images (unlearn the trigger)
4. Phase B — Fine-tune with empty labels + frozen backbone (restore clean behaviour)
5. Weight average Phase A and Phase B models
6. Run inference on test set
7. Save `submission.csv`

## 1. Install Detectron2

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.3 MB/s eta 0:00:00


## 2. Imports

In [2]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.solver import build_lr_scheduler, build_optimizer
from tqdm import tqdm

## 3. Configuration

In [3]:
POISONED_WEIGHTS = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
UNLEARN_DIR      = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set"
TEST_DIR         = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"
# ── Output paths ──
OUTPUT_DIR_A    = "/kaggle/working/phase_a_ga"      # gradient ascent checkpoint
OUTPUT_DIR_B    = "/kaggle/working/phase_b_ft"      # fine-tune checkpoint
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# ── Model architecture (MUST match poisoned model training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Unlearning hyperparameters ──
# Phase A: gradient ascent — pushes model AWAY from poisoned outputs
GA_LR    = 5e-5   # small lr to avoid exploding weights
GA_ITERS = 30     # short — just enough to disrupt the poison signal

# Phase B: empty-label fine-tune with frozen backbone — restores clean-like head
FT_LR    = 1e-4
FT_ITERS = 150    # more iterations than baseline (baseline=20)

BATCH_SIZE = 4

# ── Weight averaging mix ──
# 0.0 = pure fine-tune, 1.0 = pure gradient-ascent; 0.3 works well empirically
GA_WEIGHT_MIX = 0.3

# ── Inference ──
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

for d in [OUTPUT_DIR_A, OUTPUT_DIR_B]:
    Path(d).mkdir(parents=True, exist_ok=True)

## 4. Register unlearn dataset

In [4]:
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    base_dicts = [
        {
            "file_name":   str(Path(unlearn_dir) / im["file_name"]),
            "height":      im["height"],
            "width":       im["width"],
            "image_id":    im["id"],
            "annotations": [],
        }
        for im in coco["images"]
    ]
    # Expand to 4× by adding all flip variants: 0=original, 1=hflip, 2=vflip, 3=both
    dicts = [
        {**d, "flip": flip, "image_id": d["image_id"] * 4 + flip}
        for d in base_dicts
        for flip in (0, 1, 2, 3)
    ]
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn set: {len(base_dicts)} images × 4 flips = {len(dicts)} entries")
    return dicts

unlearn_dicts = register_unlearn(UNLEARN_DIR)


Registered unlearn set: 20 images × 4 flips = 80 entries


## 5. 16-bit image handling

In [5]:
class UInt16DatasetMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 in [0,255] with empty instances.
    Applies deterministic flips based on the 'flip' key in the dataset dict:
      0 = original, 1 = hflip, 2 = vflip, 3 = both
    """
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)
        flip = dataset_dict.get("flip", 0)
        if flip in (1, 3):  # horizontal flip
            image = image[:, ::-1, :].copy()
        if flip in (2, 3):  # vertical flip
            image = image[::-1, :, :].copy()
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


## 6. Build base config helper

In [6]:
def make_cfg(weights_path, output_dir, lr, max_iter):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                       = weights_path
    cfg.MODEL.RETINANET.NUM_CLASSES         = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES        = ANCHOR_SIZES
    cfg.DATASETS.TRAIN                      = (UNLEARN_DATASET,)
    cfg.DATASETS.TEST                       = ()
    cfg.DATALOADER.NUM_WORKERS              = 2
    cfg.SOLVER.IMS_PER_BATCH                = BATCH_SIZE
    cfg.SOLVER.BASE_LR                      = lr
    cfg.SOLVER.MAX_ITER                     = max_iter
    cfg.SOLVER.STEPS                        = []
    cfg.SOLVER.WARMUP_ITERS                 = max(1, max_iter // 10)
    cfg.SOLVER.SCHEDULER_NAME              = "WarmupCosineLR"
    cfg.OUTPUT_DIR                          = output_dir
    return cfg

## 7. Phase A — Gradient Ascent

We **maximise** the classification loss on the poisoned images.
This actively pushes model weights away from the poisoned decision boundary.
We only update the **head** (backbone frozen) to avoid destroying feature extraction.

In [7]:
print("=" * 60)
print("PHASE A: Gradient Ascent (disrupting poison signal)")
print("=" * 60)

cfg_a = make_cfg(POISONED_WEIGHTS, OUTPUT_DIR_A, GA_LR, GA_ITERS)

# Build model and load poisoned weights
model_a = build_model(cfg_a)
DetectionCheckpointer(model_a).load(POISONED_WEIGHTS)
model_a.train()

# ── Freeze backbone + FPN, only head updates ──
frozen, trainable = [], []
for name, param in model_a.named_parameters():
    if "backbone" in name or "fpn" in name:
        param.requires_grad = False
        frozen.append(name)
    else:
        trainable.append(name)
print(f"Frozen layers : {len(frozen)}")
print(f"Trainable layers: {len(trainable)}")

optimizer_a = torch.optim.SGD(
    [p for p in model_a.parameters() if p.requires_grad],
    lr=GA_LR, momentum=0.9, weight_decay=1e-4
)

mapper_a = UInt16DatasetMapper(cfg_a, is_train=True, augmentations=[])
loader_a = build_detection_train_loader(cfg_a, mapper=mapper_a, dataset=unlearn_dicts)
loader_iter_a = iter(loader_a)

model_a = model_a.cuda()

from detectron2.utils.events import EventStorage

with EventStorage() as storage:
    for i in range(GA_ITERS):
        storage.step()
        batch = next(loader_iter_a)
        optimizer_a.zero_grad()
        loss_dict = model_a(batch)
        # Gradient ASCENT: negate the classification loss
        loss = -loss_dict["loss_cls"]
        loss.backward()
        # Clip gradients to prevent instability
        torch.nn.utils.clip_grad_norm_(
            [p for p in model_a.parameters() if p.requires_grad], max_norm=1.0
        )
        optimizer_a.step()
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  GA iter {i+1:3d}/{GA_ITERS}  loss_cls (negated): {loss.item():.4f}")

# Save Phase A checkpoint
torch.save(model_a.state_dict(), Path(OUTPUT_DIR_A) / "model_ga.pth")
print("Phase A complete. Checkpoint saved.")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


PHASE A: Gradient Ascent (disrupting poison signal)
Frozen layers : 69
Trainable layers: 20


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  GA iter   1/30  loss_cls (negated): -0.0410
  GA iter  10/30  loss_cls (negated): -0.0869
  GA iter  20/30  loss_cls (negated): -0.2912
  GA iter  30/30  loss_cls (negated): -0.3823
Phase A complete. Checkpoint saved.


## 8. Phase B — Empty-label Fine-tuning with Frozen Backbone

Standard fine-tune (like the baseline) but:
- Starting from the GA-disrupted weights
- Backbone frozen throughout
- 150 iterations with cosine LR decay (vs baseline's 20 iterations)

In [8]:
print("=" * 60)
print("PHASE B: Empty-label fine-tuning (frozen backbone)")
print("=" * 60)

# We start Phase B from the GA weights
GA_CHECKPOINT = str(Path(OUTPUT_DIR_A) / "model_ga.pth")


class FrozenBackboneUnlearnTrainer(DefaultTrainer):
    """Fine-tunes only the detection head with empty labels."""

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        for name, param in model.named_parameters():
            if "backbone" in name or "fpn" in name:
                param.requires_grad = False
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable params: {n_trainable:,}")
        return model

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=unlearn_dicts)


cfg_b = make_cfg(GA_CHECKPOINT, OUTPUT_DIR_B, FT_LR, FT_ITERS)

trainer = FrozenBackboneUnlearnTrainer(cfg_b)
trainer.resume_or_load(resume=False)
trainer.train()

print("Phase B complete.")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


PHASE B: Empty-label fine-tuning (frozen backbone)
[04/29 22:45:10 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_fea

2026-04-29 22:45:26.779312: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777502727.219420      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777502727.342223      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777502728.355559      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777502728.355591      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777502728.355594      23 computation_placer.cc:177] computation placer alr

[04/29 22:46:09 d2.utils.events]:  eta: 0:01:03  iter: 39  total_loss: 0.01838  loss_cls: 0.01838  loss_box_reg: 0    time: 0.5749  last_time: 0.5756  data_time: 0.0150  last_data_time: 0.0130   lr: 0.0001  max_mem: 1712M
[04/29 22:46:21 d2.utils.events]:  eta: 0:00:51  iter: 59  total_loss: 0.008285  loss_cls: 0.008285  loss_box_reg: 0    time: 0.5780  last_time: 0.5873  data_time: 0.0151  last_data_time: 0.0158   lr: 0.0001  max_mem: 1712M
[04/29 22:46:32 d2.utils.events]:  eta: 0:00:40  iter: 79  total_loss: 0.006753  loss_cls: 0.006753  loss_box_reg: 0    time: 0.5809  last_time: 0.5836  data_time: 0.0152  last_data_time: 0.0130   lr: 0.0001  max_mem: 1712M
[04/29 22:46:44 d2.utils.events]:  eta: 0:00:29  iter: 99  total_loss: 0.005408  loss_cls: 0.005408  loss_box_reg: 0    time: 0.5845  last_time: 0.5975  data_time: 0.0159  last_data_time: 0.0131   lr: 0.0001  max_mem: 1712M
[04/29 22:46:56 d2.utils.events]:  eta: 0:00:17  iter: 119  total_loss: 0.003529  loss_cls: 0.003529  loss

## 9. Weight Averaging

Blend the Phase A (GA) and Phase B (fine-tuned) weights.
This stabilises outputs — GA alone can be noisy, FT alone may under-suppress the poison.
Mix ratio: 30% GA + 70% FT (controlled by `GA_WEIGHT_MIX`).

In [9]:
print("Averaging weights...")

weights_ga = torch.load(Path(OUTPUT_DIR_A) / "model_ga.pth", map_location="cpu")
weights_ft = torch.load(Path(OUTPUT_DIR_B) / "model_final.pth", map_location="cpu")

# Detectron2 checkpoints sometimes nest under 'model'
if "model" in weights_ft:
    weights_ft = weights_ft["model"]

averaged = {}
for key in weights_ft:
    if key in weights_ga and weights_ga[key].shape == weights_ft[key].shape:
        averaged[key] = (
            GA_WEIGHT_MIX * weights_ga[key].float()
            + (1 - GA_WEIGHT_MIX) * weights_ft[key].float()
        )
    else:
        # Key only in FT (e.g. head added) — use FT weights as-is
        averaged[key] = weights_ft[key]

AVERAGED_WEIGHTS = "/kaggle/working/model_averaged.pth"
torch.save({"model": averaged}, AVERAGED_WEIGHTS)
print(f"Averaged checkpoint saved to {AVERAGED_WEIGHTS}")

Averaging weights...
Averaged checkpoint saved to /kaggle/working/model_averaged.pth


## 10. Inference & Submission

In [10]:
cfg_infer = make_cfg(AVERAGED_WEIGHTS, OUTPUT_DIR_B, FT_LR, FT_ITERS)
cfg_infer.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
predictor = DefaultPredictor(cfg_infer)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])

    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[04/29 22:47:16 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/model_averaged.pth ...
Running inference on 2000 images...


Inference: 100%|██████████| 2000/2000 [04:14<00:00,  7.85it/s]

Wrote /kaggle/working/submission.csv  (2000 rows)


,id,image_id,prediction_string
0,0,0,0.285052 893.96 186.20 9.06 40.97 0.236229 208...
1,1,1,
2,2,10,0.275185 4.98 17.87 37.53 58.97
3,3,100,
4,4,1000,
